In [ ]:
# Import path configuration
# This provides all directory paths with automatic cluster/local detection
import sys
sys.path.insert(0, '.')
from python_config_template import (
    SPATIAL_BASE_DIR, SHARED_DIR, PLOTS_DIR,
    RAW_DATA_DIR, H5AD_OUTPUT_DIR, PAPER_H5AD_DIR,
    FINAL_OBJ_DIR, CELL_TYPES_DIR,
    TOPICS_DATA_PATH, SPATIAL_PLOTS_DIR,
    ENRICHMENT_PROGRAMS_DIR, ENRICHMENT_PATHWAYS_DIR,
    CASE_SAMPLES, CONTROL_SAMPLES, SAMPLE_METADATA,
    ALL_ANNOTATED_PATH,
    moderate_list, advanced_list, severe_list,
    get_gef_path
)


In [ ]:
# Import libraries for spatial transcriptomics data processing
import stereo as st
import os
import gc
import numpy as np
import pandas as pd
from stereo.utils.data_helper import split
import numpy as np
from importlib import reload
import scanpy as sc
import anndata as ad


In [ ]:
def data_process(adata):
    adata.var['mt'] = adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]
    sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])
    sc.pp.scale(adata, max_value=10, zero_center=False)
    sc.tl.pca(adata, svd_solver='arpack')
    sc.external.pp.harmony_integrate(adata, "sample")
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40, use_rep= "X_pca_harmony")
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution = 0.1)
    return adata


In [ ]:
print(f"Using paths from configuration:")
print(f"  Raw data: {RAW_DATA_DIR}")
print(f"  Output: {H5AD_OUTPUT_DIR}")

In [ ]:
# Sample lists are imported from config: CASE_SAMPLES and CONTROL_SAMPLES

# Process case samples: read GEF files, calculate QC, add metadata, and save as H5AD
for sample in CASE_SAMPLES:
    data_path = get_gef_path(sample, RAW_DATA_DIR)
    tmp = st.io.read_gef(file_path=data_path, bin_type='cell_bins')
    tmp.tl.cal_qc()
    tmp.tl.raw_checkpoint()
    tmp.cells["sample_id"] = sample
    tmp.cells["diagnosis"] = "case"
    globals()[sample] = tmp
    st.io.stereo_to_anndata(tmp, flavor='scanpy', output=os.path.join(H5AD_OUTPUT_DIR, f"{sample}.h5ad"))

# Process control samples: same steps as case samples but labeled as "control"
for sample in CONTROL_SAMPLES:
    data_path = get_gef_path(sample, RAW_DATA_DIR)
    tmp = st.io.read_gef(file_path=data_path, bin_type='cell_bins')
    tmp.tl.cal_qc()
    tmp.tl.raw_checkpoint()
    tmp.cells["sample_id"] = sample
    tmp.cells["diagnosis"] = "control"
    globals()[sample] = tmp
    st.io.stereo_to_anndata(tmp, flavor='scanpy', output=os.path.join(H5AD_OUTPUT_DIR, f"{sample}.h5ad"))

In [ ]:
# Load preprocessed H5AD files and annotate with metadata
# Using H5AD_OUTPUT_DIR from config

# Sample lists defined by severity level
control_list = CONTROL_SAMPLES  # From config


###### Control
for num, files in enumerate(control_list):
    data_path = os.path.join(H5AD_OUTPUT_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "control"
    data.obs["sample"] = files
    data.obs["levels"] = "control"
    globals()[files] = data
    print(files)

###### moderate_list
for num, files in enumerate(moderate_list):
    data_path = os.path.join(H5AD_OUTPUT_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "case"
    data.obs["sample"] = files
    data.obs["levels"] = "moderate"
    globals()[files] = data
    print(files)

##### advanced_list
for num, files in enumerate(advanced_list):
    data_path = os.path.join(H5AD_OUTPUT_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "case"
    data.obs["sample"] = files
    data.obs["levels"] = "advanced"
    globals()[files] = data
    print(files)

##### severe_list
for num, files in enumerate(severe_list):
    data_path = os.path.join(H5AD_OUTPUT_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "case"
    data.obs["sample"] = files
    data.obs["levels"] = "severe"
    globals()[files] = data
    print(files)




In [ ]:
# NOTE: Duplicate of cell 3 - processes same samples again (for re-running/updating)
# Using CASE_SAMPLES and CONTROL_SAMPLES from config
# TODO: ask Gali if we can delete this cell
# Process case samples: read GEF, calculate QC, add metadata, save as H5AD
for sample in CASE_SAMPLES:
    data_path = get_gef_path(sample, RAW_DATA_DIR)
    tmp = st.io.read_gef(file_path=data_path, bin_type='cell_bins')
    tmp.tl.cal_qc()
    tmp.tl.raw_checkpoint()
    tmp.cells["sample_id"] = sample
    tmp.cells["diagnosis"] = "case"
    globals()[sample] = tmp
    st.io.stereo_to_anndata(tmp, flavor='scanpy', output=os.path.join(H5AD_OUTPUT_DIR, f"{sample}.h5ad"))

# Process control samples: same steps as case samples
for sample in CONTROL_SAMPLES:
    data_path = get_gef_path(sample, RAW_DATA_DIR)
    tmp = st.io.read_gef(file_path=data_path, bin_type='cell_bins')
    tmp.tl.cal_qc()
    tmp.tl.raw_checkpoint()
    tmp.cells["sample_id"] = sample
    tmp.cells["diagnosis"] = "control"
    globals()[sample] = tmp
    st.io.stereo_to_anndata(tmp, flavor='scanpy', output=os.path.join(H5AD_OUTPUT_DIR, f"{sample}.h5ad"))

In [ ]:
# Load samples from paper-specific directory (different from cell 4)
# Using PAPER_H5AD_DIR from config

# Sample lists defined by severity level
control_list = CONTROL_SAMPLES  # From config


###### Control
# import the stereo data
for num, files in enumerate(control_list):
    data_path = os.path.join(PAPER_H5AD_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "control"
    data.obs["sample"] = files
    data.obs["levels"] = "control"
    globals()[files] = data
    print(files)

###### moderate_list
# import the stereo data
for num, files in enumerate(moderate_list):
    data_path = os.path.join(PAPER_H5AD_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "case"
    data.obs["sample"] = files
    data.obs["levels"] = "moderate"
    globals()[files] = data
    print(files)

##### advanced_list
# import the stereo data
for num, files in enumerate(advanced_list):
    data_path = os.path.join(PAPER_H5AD_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "case"
    data.obs["sample"] = files
    data.obs["levels"] = "advanced"
    globals()[files] = data
    print(files)

##### severe_list
# import the stereo data
for num, files in enumerate(severe_list):
    data_path = os.path.join(PAPER_H5AD_DIR, f"{files}.h5ad")
    data = sc.read_h5ad(data_path)
    data.obs["diagnosis"] = "case"
    data.obs["sample"] = files
    data.obs["levels"] = "severe"
    globals()[files] = data
    print(files)


In [ ]:
# Concatenate all 12 samples into a single AnnData object for integrated analysis
merge = ad.concat(
    [B01806B5, B01806B6, B01809A3, B01809A4, 
     D02175A4, D02175A6, A02092E1, B02009F6, 
     C02248B5, B02008C6, B02008D2, B01809C2],
    join = "outer"
)
merge.obs_names_make_unique()
merge = merge[merge.obs["n_genes_by_counts"]>=150]
merge.raw = merge

merge = data_process(merge)

# Optional: save intermediate integrated object
merge.write_h5ad(os.path.join(H5AD_OUTPUT_DIR, "integrated_sc.h5ad"))

In [ ]:
# Re-run clustering at higher resolution (0.3) for finer cell type separation
sc.tl.leiden(merge, resolution=0.3)

# Visualize clusters on UMAP
sc.pl.umap(merge, color=["leiden"])

In [ ]:
# Cell type annotation using known marker genes for brain cell types
marker_genes_dict = {
    'neurons':["MEG3", "GRIN1", "GRIN2A", "RBFOX3"],
    'Ex': ["SLC17A7", "SLC17A6", "CAMK2A", "RORB", "TOX","FOXP2", "CUX2", "LINC00507",
                    "MAML2", "PROX1"],
    'End': ['CLDN5', 'FLT1'],
    'Mic': ["CX3CR1", "P2RY12", "TMEM119", "ITGAM","CD74","CSF1R","SPP1"],
    'Inh': ["GAD1", "GAD2","SST","PVALB","VIP", "KIT"],
    'Ast': ["GFAP", "AQP4", "SLC1A3", "ALDH1L1"],
    'Oli': ['MBP',"MOG","PLP1"],
    'Opc': ['PDGFRA', 'VCAN']
}
sc.pl.dotplot(merge, marker_genes_dict, 'leiden', dendrogram=True)

# Map cluster IDs to cell type names based on marker expression patterns
cell_types = {
    "7": "OPC", "1": "OLIGO", "2": "AST", "3": "INH", 
    "4": "END", "6": "Microglia", "5": "other", "0": "Ex"
}

# perform the annotation
# annotate the cluster

merge.obs['annotation'] = merge.obs['leiden'].astype(str).map(cell_types)
sc.pl.dotplot(merge, marker_genes_dict, 'annotation', dendrogram=True)


sc.pl.umap(merge, color = "annotation")

In [ ]:
# Save final annotated dataset with all samples integrated, processed, and annotated
# Using H5AD_OUTPUT_DIR from config
merge.write(ALL_ANNOTATED_PATH, compression="gzip")